In [ ]:
from google.colab import drive
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/CREMA_5Channel_Tensors', exist_ok=True)

Mounted at /content/drive


In [ ]:
from google.colab import files
import os

files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d ejlok1/cremad

!mkdir -p /content/cremad_raw
!unzip -q cremad.zip -d /content/cremad_raw

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/ejlok1/cremad
License(s): ODC Attribution License (ODC-By)
100% 451M/451M [00:02<00:00, 198MB/s]



In [ ]:
import librosa
import numpy as np
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/CREMA_5CH_TENSORS'
os.makedirs(SAVE_DIR, exist_ok=True)

def extract_5ch_tensor(file_path, target_frames=94):
    y, sr = librosa.load(file_path, duration=3.0, sr=22050)

    if len(y) < sr * 3:
        y = np.pad(y, (0, sr * 3 - len(y)))
    else:
        y = y[:sr * 3]

    # Mel Spectrogram (128 bins)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    ch1 = librosa.power_to_db(mel, ref=np.max)

    # MFCC (128 coefficients)
    ch2 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=128)

    # Chroma (Pitch-based, 128 bins via interpolation)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=12)
    # We pad/resize to 128 to match the CNN height
    ch3 = np.tile(chroma, (11, 1))[:128, :]

    # Spectral Contrast
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)
    ch4 = np.tile(contrast, (19, 1))[:128, :]

    # Tonnetz
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    ch5 = np.tile(tonnetz, (22, 1))[:128, :]

    # Stack into (Height, Time, Channels) -> (128, 94+, 5)
    stack = np.stack([ch1, ch2, ch3, ch4, ch5], axis=-1)

    # Trim time dimension to exactly 94 frames
    return stack[:, :target_frames, :].astype(np.float32)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def extract_high_fidelity_5ch(file_path, target_shape=(128, 94)):
    # Load audio (fixed to 3 seconds for consistency)
    y, sr = librosa.load(file_path, duration=3.0)

    # Padding/clipping to keep exactly 3 secs
    if len(y) < sr * 3:
        y = np.pad(y, (0, sr * 3 - len(y)))
    else:
        y = y[:sr * 3]

    # Mel-Spectrogram (Energy)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    ch1 = librosa.power_to_db(mel, ref=np.max)

    # MFCCs (Timbre/Texture)
    ch2 = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=128)

    # Chroma STFT (Harmonic Pitch)
    ch3 = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=128)

    # Spectral Contrast (Clarity/Brightness)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=6)
    ch4 = np.pad(contrast, ((0, 121), (0, 0)), mode='constant')

    # Tonnetz (Tonal Relationships), Captures the "emotional dissonance"
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    ch5 = np.pad(tonnetz, ((0, 122), (0, 0)), mode='constant')

    stack = np.stack([ch1, ch2, ch3, ch4, ch5], axis=-1)
    stack = stack[:, :target_shape[1], :]

    return stack.astype(np.float32)

In [ ]:
RAW_AUDIO_PATH = '/content/cremad_raw/AudioWAV'
all_files = [f for f in os.listdir(RAW_AUDIO_PATH) if f.endswith('.wav')]

print(f"Processing {len(all_files)} files into 5-channel tensors...")

for fname in tqdm(all_files):
    output_path = os.path.join(SAVE_DIR, fname.replace('.wav', '.npy'))

    if not os.path.exists(output_path):
        try:
            tensor = extract_5ch_tensor(os.path.join(RAW_AUDIO_PATH, fname))
            np.save(output_path, tensor)
        except Exception as e:
            print(f"Error on {fname}: {e}")

print(f"Done! Tensors are saved in: {SAVE_DIR}")

Processing 7442 files into 5-channel tensors...


 96%|█████████▌| 7155/7442 [47:04<01:40,  2.84it/s]/usr/local/lib/python3.12/dist-packages/librosa/core/pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
100%|██████████| 7442/7442 [49:00<00:00,  2.53it/s]

Done! Tensors are saved in: /content/drive/MyDrive/CREMA_5CH_TENSORS


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_5ch_fusion_model(input_shape=(128, 94, 5)):
    inputs = layers.Input(shape=input_shape)

    # Input Normalization (Crucial for 5 different feature scales)
    x = layers.BatchNormalization()(inputs)

    # Feature Extraction
    x = layers.Conv2D(64, (3, 3), padding='same', activation='swish')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', activation='swish')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    # Deep Texture Analysis
    x = layers.Conv2D(128, (3, 3), padding='same', activation='swish')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, (1, 1), activation='swish')(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.BatchNormalization()(x)

    # 6 Emotions: SAD, ANG, HAP, FEA, DIS, NEU
    outputs = layers.Dense(6, activation='softmax')(x)

    model = models.Model(inputs, outputs, name="CREMA_5CH_FUSION")
    return model

model = build_5ch_fusion_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "CREMA_5CH_FUSION"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 94, 5)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 94, 5)     │            20 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 94, 64)    │         2,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 94, 64)    │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 47, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 47, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 47, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 47, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 23, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 23, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 23, 256)    │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 412,762 (1.57 MB)

 Trainable params: 411,984 (1.57 MB)

 Non-trainable params: 778 (3.04 KB)

In [ ]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

tensor_path = '/content/drive/MyDrive/CREMA_5CH_TENSORS'
filenames = os.listdir(tensor_path)

data = []
for f in filenames:
    if f.endswith('.npy'):
        parts = f.split('_')
        if len(parts) >= 3:
            emotion = parts[2]
            data.append({'filename': f, 'emotion': emotion})

df = pd.DataFrame(data)

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['emotion'], random_state=42)

print(f"Training on {len(train_df)} files, Validating on {len(val_df)} files.")

Training on 5953 files, Validating on 1489 files.


In [ ]:
class CREMA5ChGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, base_path, batch_size=32, shuffle=True):
        self.df = dataframe
        self.base_path = base_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.labels = {'SAD':0, 'ANG':1, 'HAP':2, 'FEA':3, 'DIS':4, 'NEU':5}
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            self.df = self.df.sample(frac=1).reset_index(drop=True)

    def __getitem__(self, index):
        batch_df = self.df[index*self.batch_size : (index+1)*self.batch_size]
        X, y = [], []
        for _, row in batch_df.iterrows():
            file_path = f"{self.base_path}/{row['filename'].replace('.wav', '.npy')}"
            tensor = np.load(file_path)
            X.append(tensor)
            y.append(self.labels[row['emotion']])
        return np.array(X), np.array(y)
train_gen = CREMA5ChGenerator(train_df, '/content/drive/MyDrive/CREMA_5CH_TENSORS', shuffle=True)
val_gen = CREMA5ChGenerator(val_df, '/content/drive/MyDrive/CREMA_5CH_TENSORS', shuffle=False)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler
import math

def cosine_decay_with_warmup(epoch, lr):
    warmup_epochs = 5
    max_lr = 1e-3
    min_lr = 1e-5
    total_epochs = 50

    if epoch < warmup_epochs:
        return max_lr * (epoch + 1) / warmup_epochs
    else:
        # Cosine decay
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

lr_callback = LearningRateScheduler(cosine_decay_with_warmup)

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/CREMA_5CH_Best_GPU.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[checkpoint, early_stop, lr_callback]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 414ms/step - accuracy: 0.2438 - loss: 2.0190
Epoch 1: val_accuracy improved from None to 0.27378, saving model to /content/drive/MyDrive/CREMA_5CH_Best_GPU.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CREMA_5CH_Best_GPU.keras
186/186 ━━━━━━━━━━━━━━━━━━━━ 258s 508ms/step - accuracy: 0.2797 - loss: 1.8475 - val_accuracy: 0.2738 - val_loss: 1.7468 - learning_rate: 2.0000e-04
Epoch 2/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.3299 - loss: 1.6288
Epoch 2: val_accuracy did not improve from 0.27378
186/186 ━━━━━━━━━━━━━━━━━━━━ 29s 155ms/step - accuracy: 0.3385 - loss: 1.6102 - val_accuracy: 0.2520 - val_loss: 1.6784 - learning_rate: 4.0000e-04
Epoch 3/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.3519 - loss: 1.5835
Epoch 3: val_accuracy improved from 0.27378 to 0.42935, saving model to /content/drive/MyDrive/CREMA_5CH_Best_GPU.keras

Epoch 3: finished saving model to /content/drive/MyDrive/CREMA_5CH_

#Tuning

In [ ]:
def build_regularized_5ch_model(input_shape=(128, 94, 5)):
    inputs = layers.Input(shape=input_shape)
    x = layers.BatchNormalization()(inputs)

    # Block 1
    x = layers.Conv2D(64, (3, 3), padding='same', activation='swish')(x)
    x = layers.SpatialDropout2D(0.2)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 2
    x = layers.Conv2D(128, (3, 3), padding='same', activation='swish')(x)
    x = layers.SpatialDropout2D(0.3)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 3
    x = layers.Conv2D(256, (1, 1), activation='swish')(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.6)(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(6, activation='softmax')(x)

    return models.Model(inputs, outputs)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_regularized_5ch_model(input_shape=(128, 94, 5)):
    inputs = layers.Input(shape=input_shape)

    # Normalization
    x = layers.BatchNormalization()(inputs)

    # Block 1: Lower Level Textures
    x = layers.Conv2D(64, (3, 3), padding='same', activation='swish')(x)
    x = layers.SpatialDropout2D(0.2)(x) # Drops entire feature maps
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 2: Mid-Level Emotional Cues
    x = layers.Conv2D(128, (3, 3), padding='same', activation='swish')(x)
    x = layers.SpatialDropout2D(0.3)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 3: Fusion
    x = layers.Conv2D(256, (1, 1), activation='swish')(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Heavy Regularization Head
    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.6)(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.BatchNormalization()(x)

    outputs = layers.Dense(6, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

model = build_regularized_5ch_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
import numpy as np

class CREMA5ChAugmentedGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, base_path, batch_size=32, shuffle=True, augment=False):

        super().__init__()
        self.df = dataframe
        self.base_path = base_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.labels = {'SAD':0, 'ANG':1, 'HAP':2, 'FEA':3, 'DIS':4, 'NEU':5}
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            self.df = self.df.sample(frac=1).reset_index(drop=True)

    def __getitem__(self, index):
        batch_df = self.df[index*self.batch_size : (index+1)*self.batch_size]
        X, y = [], []

        for _, row in batch_df.iterrows():
            file_path = f"{self.base_path}/{row['filename']}"
            tensor = np.load(file_path)

            if self.augment:
                #Frequency Masking (Up to 10 rows)
                f_width = np.random.randint(0, 10)
                f_start = np.random.randint(0, 128 - f_width)
                tensor[f_start:f_start+f_width, :, :] = 0

                #Time Masking (Up to 8 columns)
                t_width = np.random.randint(0, 8)
                t_start = np.random.randint(0, 94 - t_width)
                tensor[:, t_start:t_start+t_width, :] = 0

            X.append(tensor)
            y.append(self.labels[row['emotion']])

        return np.array(X), np.array(y)

train_gen = CREMA5ChAugmentedGenerator(train_df, tensor_path, augment=True, shuffle=True)
val_gen = CREMA5ChAugmentedGenerator(val_df, tensor_path, augment=False, shuffle=False)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    '/content/drive/MyDrive/CREMA_5CH_Final_60Pct.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max'
)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[checkpoint, lr_callback, early_stop]
)

Epoch 1/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 40s 161ms/step - accuracy: 0.2176 - loss: 1.8109 - val_accuracy: 0.2425 - val_loss: 1.7735 - learning_rate: 2.0000e-04
Epoch 2/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 29s 155ms/step - accuracy: 0.2653 - loss: 1.7199 - val_accuracy: 0.2106 - val_loss: 1.7201 - learning_rate: 4.0000e-04
Epoch 3/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 30s 159ms/step - accuracy: 0.2984 - loss: 1.6563 - val_accuracy: 0.3336 - val_loss: 1.5788 - learning_rate: 6.0000e-04
Epoch 4/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 30s 161ms/step - accuracy: 0.3204 - loss: 1.6151 - val_accuracy: 0.3465 - val_loss: 1.5431 - learning_rate: 8.0000e-04
Epoch 5/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 29s 154ms/step - accuracy: 0.3357 - loss: 1.6064 - val_accuracy: 0.3961 - val_loss: 1.4674 - learning_rate: 0.0010
Epoch 6/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 152ms/step - accuracy: 0.3375 - loss: 1.5853 - val_accuracy: 0.3743 - val_loss: 1.4839 - learning_rate: 0.0010
Epoch 7/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 151ms/step

In [ ]:
import tensorflow as tf

model_path = '/content/drive/MyDrive/CREMA_5CH_Best_GPU.keras'
if os.path.exists(model_path):
    model = tf.keras.models.load_model(model_path)
    print("Successfully loaded the 46% accuracy model. Resuming...")
else:
    print("Checkpoint not found. We may need to re-run the build_model cell.")

#Lowering the Learning Rate slightly for "Fine-Tuning"
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_resume = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    initial_epoch=12,
    callbacks=[checkpoint, early_stop]
)

Successfully loaded the 46% accuracy model. Resuming...
Epoch 13/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 39s 162ms/step - accuracy: 0.4746 - loss: 1.3608 - val_accuracy: 0.5190 - val_loss: 1.2159
Epoch 14/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 29s 153ms/step - accuracy: 0.5160 - loss: 1.2262 - val_accuracy: 0.5034 - val_loss: 1.2679
Epoch 15/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 152ms/step - accuracy: 0.5296 - loss: 1.2035 - val_accuracy: 0.5170 - val_loss: 1.1987
Epoch 16/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 152ms/step - accuracy: 0.5386 - loss: 1.1947 - val_accuracy: 0.4817 - val_loss: 1.2822
Epoch 17/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 150ms/step - accuracy: 0.5432 - loss: 1.1778 - val_accuracy: 0.5136 - val_loss: 1.2045
Epoch 18/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 152ms/step - accuracy: 0.5512 - loss: 1.1558 - val_accuracy: 0.5170 - val_loss: 1.1936
Epoch 19/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 151ms/step - accuracy: 0.5467 - loss: 1.1637 - val_accuracy: 0.5177 - val_loss: 1.2194
Epoch 20/50
186/186 

In [ ]:
def squeeze_excitation_block(input_tensor, ratio=8):
    channels = input_tensor.shape[-1]

    # Squeeze, Global Average Pooling
    se = layers.GlobalAveragePooling2D()(input_tensor)

    # Excitation, Two Dense layers to learn channel importance
    se = layers.Dense(channels // ratio, activation='relu', kernel_initializer='he_normal', use_bias=False)(se)
    se = layers.Dense(channels, activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(se)

    se = layers.Reshape((1, 1, channels))(se)
    return layers.multiply([input_tensor, se])

def build_attention_5ch_model(input_shape=(128, 94, 5)):
    inputs = layers.Input(shape=input_shape)
    x = layers.BatchNormalization()(inputs)

    # Block 1 + Attention
    x = layers.Conv2D(64, (3, 3), padding='same', activation='swish')(x)
    x = squeeze_excitation_block(x)
    x = layers.SpatialDropout2D(0.2)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 2 + Attention
    x = layers.Conv2D(128, (3, 3), padding='same', activation='swish')(x)
    x = squeeze_excitation_block(x)
    x = layers.SpatialDropout2D(0.3)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 3: Fusion
    x = layers.Conv2D(256, (1, 1), activation='swish')(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.BatchNormalization()(x)

    outputs = layers.Dense(6, activation='softmax')(x)
    return models.Model(inputs, outputs)

model = build_attention_5ch_model()

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)

model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_final = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=[checkpoint]
)

Epoch 1/30
186/186 ━━━━━━━━━━━━━━━━━━━━ 37s 158ms/step - accuracy: 0.1648 - loss: 1.7973 - val_accuracy: 0.1712 - val_loss: 1.7911
Epoch 2/30
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 151ms/step - accuracy: 0.1887 - loss: 1.7876 - val_accuracy: 0.2276 - val_loss: 1.7884
Epoch 3/30
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 152ms/step - accuracy: 0.2105 - loss: 1.7789 - val_accuracy: 0.2595 - val_loss: 1.7781
Epoch 4/30
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 150ms/step - accuracy: 0.2273 - loss: 1.7667 - val_accuracy: 0.2636 - val_loss: 1.7551
Epoch 5/30
186/186 ━━━━━━━━━━━━━━━━━━━━ 27s 147ms/step - accuracy: 0.2329 - loss: 1.7543 - val_accuracy: 0.2683 - val_loss: 1.7291
Epoch 6/30
186/186 ━━━━━━━━━━━━━━━━━━━━ 27s 145ms/step - accuracy: 0.2513 - loss: 1.7333 - val_accuracy: 0.2724 - val_loss: 1.7064
Epoch 7/30
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 153ms/step - accuracy: 0.2557 - loss: 1.7177 - val_accuracy: 0.2717 - val_loss: 1.6904
Epoch 8/30
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 149ms/step - accuracy: 0.2609 - loss: 1

#Further model optimization

In [ ]:
from google.colab import drive
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

tensor_path = '/content/drive/MyDrive/CREMA_5CH_TENSORS'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class CREMA5ChAugmentedGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, base_path, batch_size=32, shuffle=True, augment=False):
        super().__init__()
        self.df = dataframe
        self.base_path = base_path
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.labels = {'SAD':0, 'ANG':1, 'HAP':2, 'FEA':3, 'DIS':4, 'NEU':5}
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            self.df = self.df.sample(frac=1).reset_index(drop=True)

    def __getitem__(self, index):
        batch_df = self.df[index*self.batch_size : (index+1)*self.batch_size]
        X, y = [], []
        for _, row in batch_df.iterrows():
            file_path = os.path.join(self.base_path, row['filename'])
            tensor = np.load(file_path)
            if self.augment:
                # Frequency Masking (Up to 10 rows)
                f_width = np.random.randint(0, 10)
                f_start = np.random.randint(0, 128 - f_width)
                tensor[f_start:f_start+f_width, :, :] = 0
                # Time Masking (Up to 8 columns)
                t_width = np.random.randint(0, 8)
                t_start = np.random.randint(0, 94 - t_width)
                tensor[:, t_start:t_start+t_width, :] = 0
            X.append(tensor)
            y.append(self.labels[row['emotion']])
        return np.array(X), np.array(y)

In [ ]:
filenames = os.listdir(tensor_path)
data = [{'filename': f, 'emotion': f.split('_')[2]} for f in filenames if f.endswith('.npy')]
df = pd.DataFrame(data)

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['emotion'], random_state=42)

train_gen = CREMA5ChAugmentedGenerator(train_df, tensor_path, augment=True, shuffle=True)
val_gen = CREMA5ChAugmentedGenerator(val_df, tensor_path, augment=False, shuffle=False)

In [ ]:
def squeeze_excitation_block(input_tensor, ratio=8):
    channels = input_tensor.shape[-1]

    #Global Average Pooling
    se = layers.GlobalAveragePooling2D()(input_tensor)

    #Two Dense layers to learn channel importance
    se = layers.Dense(channels // ratio, activation='relu', kernel_initializer='he_normal', use_bias=False)(se)
    se = layers.Dense(channels, activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(se)

    #Reshape to multiply back to the original feature map
    se = layers.Reshape((1, 1, channels))(se)
    return layers.multiply([input_tensor, se])

def build_attention_5ch_model(input_shape=(128, 94, 5)):
    inputs = layers.Input(shape=input_shape)
    x = layers.BatchNormalization()(inputs)

    #Block 1 + Attention
    x = layers.Conv2D(64, (3, 3), padding='same', activation='swish')(x)
    x = squeeze_excitation_block(x) # <--- New Attention Layer
    x = layers.SpatialDropout2D(0.2)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    #Block 2 + Attention
    x = layers.Conv2D(128, (3, 3), padding='same', activation='swish')(x)
    x = squeeze_excitation_block(x) # <--- New Attention Layer
    x = layers.SpatialDropout2D(0.3)(x)
    x = layers.MaxPooling2D((2, 2))(x)

    #Block 3: Fusion
    x = layers.Conv2D(256, (1, 1), activation='swish')(x)
    x = layers.GlobalAveragePooling2D()(x)

    #Decision Head
    x = layers.Dense(512, activation='swish')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='swish')(x)
    x = layers.BatchNormalization()(x)

    outputs = layers.Dense(6, activation='softmax')(x)
    return models.Model(inputs, outputs)

model = build_attention_5ch_model()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/CREMA_5CH_Attention_Best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [ ]:
model = build_attention_5ch_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 13s/step - accuracy: 0.2394 - loss: 1.7452 
Epoch 1: val_accuracy improved from None to 0.17255, saving model to /content/drive/MyDrive/CREMA_5CH_Attention_Best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CREMA_5CH_Attention_Best.keras
186/186 ━━━━━━━━━━━━━━━━━━━━ 2959s 16s/step - accuracy: 0.2727 - loss: 1.7013 - val_accuracy: 0.1726 - val_loss: 1.7627 - learning_rate: 0.0010
Epoch 2/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.3174 - loss: 1.6218
Epoch 2: val_accuracy improved from 0.17255 to 0.17867, saving model to /content/drive/MyDrive/CREMA_5CH_Attention_Best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/CREMA_5CH_Attention_Best.keras
186/186 ━━━━━━━━━━━━━━━━━━━━ 29s 153ms/step - accuracy: 0.3152 - loss: 1.6120 - val_accuracy: 0.1787 - val_loss: 1.7052 - learning_rate: 0.0010
Epoch 3/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.3306 - loss: 1.5933
Epoch 3: val_accuracy 

#Multi scale model to reach 60% + accuracy

field of views(small kernels for texture, large kernels for melody) to capture the emotion.

In [ ]:
def build_multiscale_attention_model(input_shape=(128, 94, 5)):
    inputs = layers.Input(shape=input_shape)
    x = layers.BatchNormalization()(inputs)

    b1 = layers.Conv2D(64, (3, 3), padding='same', activation='swish')(x)

    #Broad patterns
    b2 = layers.Conv2D(64, (3, 3), padding='same', dilation_rate=(2, 2), activation='swish')(x)

    #Concatenating the branches
    x = layers.concatenate([b1, b2])
    x = squeeze_excitation_block(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.SpatialDropout2D(0.3)(x)

    #Deep Fusion Layer
    x = layers.Conv2D(128, (3, 3), padding='same', activation='swish')(x)
    x = layers.GlobalAveragePooling2D()(x)

    #Dense Head with Label Smoothing
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(6, activation='softmax')(x)

    return models.Model(inputs, outputs)

model = build_multiscale_attention_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def squeeze_excitation_block(input_tensor, ratio=8):
    channels = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Dense(channels // ratio, activation='relu')(se)
    se = layers.Dense(channels, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, channels))(se)
    return layers.multiply([input_tensor, se])

def build_multiscale_attention_model(input_shape=(128, 94, 5)):
    inputs = layers.Input(shape=input_shape)
    x = layers.BatchNormalization()(inputs)

    b1 = layers.Conv2D(64, (3, 3), padding='same', activation='swish')(x)

    b2 = layers.Conv2D(64, (3, 3), padding='same', dilation_rate=(2, 2), activation='swish')(x)

    # Merge and Attend
    x = layers.concatenate([b1, b2])
    x = squeeze_excitation_block(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.SpatialDropout2D(0.3)(x)

    # Mid-Level Fusion
    x = layers.Conv2D(128, (3, 3), padding='same', activation='swish')(x)
    x = squeeze_excitation_block(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Global Processing
    x = layers.GlobalAveragePooling2D()(x)

    # Final Decision Head
    x = layers.Dense(256, activation='swish')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.BatchNormalization()(x)

    outputs = layers.Dense(6, activation='softmax')(x)

    return models.Model(inputs, outputs)

model = build_multiscale_attention_model()

In [ ]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

history_final = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.2410 - loss: 1.8219
Epoch 1: val_accuracy did not improve from 0.53804
186/186 ━━━━━━━━━━━━━━━━━━━━ 40s 148ms/step - accuracy: 0.2715 - loss: 1.7386 - val_accuracy: 0.2235 - val_loss: 1.7563 - learning_rate: 0.0010
Epoch 2/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.3239 - loss: 1.5954
Epoch 2: val_accuracy did not improve from 0.53804
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 149ms/step - accuracy: 0.3249 - loss: 1.5955 - val_accuracy: 0.2575 - val_loss: 1.7082 - learning_rate: 0.0010
Epoch 3/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.3504 - loss: 1.5631
Epoch 3: val_accuracy did not improve from 0.53804

Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
186/186 ━━━━━━━━━━━━━━━━━━━━ 27s 145ms/step - accuracy: 0.3535 - loss: 1.5538 - val_accuracy: 0.3689 - val_loss: 1.5419 - learning_rate: 0.0010
Epoch 4/50
186/186 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.3755 -

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True
)

callbacks = [early_stop, reduce_lr, checkpoint]

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_improved_model(input_shape, num_classes):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=100,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 28s 149ms/step - accuracy: 0.3318 - loss: 1.5994 - val_accuracy: 0.2507 - val_loss: 1.7090 - learning_rate: 1.2500e-04
Epoch 2/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 27s 147ms/step - accuracy: 0.3513 - loss: 1.5622 - val_accuracy: 0.3336 - val_loss: 1.5988 - learning_rate: 1.2500e-04
Epoch 3/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 27s 144ms/step - accuracy: 0.3495 - loss: 1.5569 - val_accuracy: 0.3818 - val_loss: 1.4967 - learning_rate: 1.2500e-04
Epoch 4/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 26s 141ms/step - accuracy: 0.3528 - loss: 1.5415 - val_accuracy: 0.3607 - val_loss: 1.4887 - learning_rate: 1.2500e-04
Epoch 5/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 27s 142ms/step - accuracy: 0.3626 - loss: 1.5389 - val_accuracy: 0.3852 - val_loss: 1.4640 - learning_rate: 1.2500e-04
Epoch 6/100
186/186 ━━━━━━━━━━━━━━━━━━━━ 27s 143ms/step - accuracy: 0.3730 - loss: 1.5277 - val_accuracy: 0.3859 - val_loss: 1.4625 - learning_rate: 1.2500e-04
Epoch 7/100
186/186 ━━━━━━━━━━━━━━━━━━━━